# Warli Motif Generation (DCGAN)

This notebook provides an end-to-end workflow:

1. Verify dataset structure
2. Train DCGAN (or load checkpoint)
3. Generate samples
4. Run evaluation: SSIM, Symmetry, FID, Diversity

**Expected dataset structure**:
```
data/warli_dataset/
  man/
    *.png / *.jpg
```


In [ ]:
# --- 0) Imports & paths ---
import os, glob, random, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# Make repo imports work (run from notebooks/)
import sys
sys.path.append('..')

from models.dcgan import DCGANGenerator, DCGANDiscriminator

DATA_ROOT = '../data/warli_dataset'  # must contain class folder 'man'
OUT_DIR = '../results/notebook_runs'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'samples'), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'final_1000'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
# --- 1) Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('Seed set to', SEED)


In [ ]:
# --- 2) Dataset check ---
assert os.path.exists(DATA_ROOT), f'Missing {DATA_ROOT}'
classes = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))]
print('Classes found:', classes)
assert 'man' in classes, "Expected class folder 'man'"
print('Num images in man:', len(glob.glob(os.path.join(DATA_ROOT, 'man', '*'))))


In [ ]:
# --- 3) Training config (edit here) ---
IMAGE_SIZE = 128   # 64 or 128
BATCH_SIZE = 64
NZ = 100
NGF = 64
NDF = 64
LR = 2e-4
BETAS = (0.5, 0.999)

# Notebook default: shorter run for demo; set higher for full training
NUM_EPOCHS = 50
SAVE_EPOCHS = {1, 10, 25, 50}

transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.ImageFolder(DATA_ROOT, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
print('Dataset size:', len(dataset))


In [ ]:
# --- 4) Build models ---
G = DCGANGenerator(nz=NZ, ngf=NGF, nc=1, image_size=IMAGE_SIZE).to(device)
# Use logits (sigmoid=False) + BCEWithLogitsLoss (recommended)
D = DCGANDiscriminator(ndf=NDF, nc=1, image_size=IMAGE_SIZE, sigmoid=False).to(device)

criterion = nn.BCEWithLogitsLoss()
optG = optim.Adam(G.parameters(), lr=LR, betas=BETAS)
optD = optim.Adam(D.parameters(), lr=LR, betas=BETAS)

fixed_noise = torch.randn(25, NZ, 1, 1, device=device)

print('Models ready.')


In [ ]:
# --- 5) Training loop (DCGAN) ---
G_losses, D_losses = [], []

def save_grid(epoch: int):
    G.eval()
    with torch.no_grad():
        fake = G(fixed_noise).detach().cpu()
    grid = vutils.make_grid(fake, nrow=5, padding=2, normalize=True)
    outp = os.path.join(OUT_DIR, 'samples', f'samples_epoch_{epoch:04d}.png')
    vutils.save_image(grid, outp)
    print('Saved:', outp)

print('Starting training...')
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    G.train(); D.train()

    lossG_epoch = None
    lossD_epoch = None

    for real, _ in loader:
        real = real.to(device)
        b = real.size(0)

        real_y = torch.ones(b, device=device)
        fake_y = torch.zeros(b, device=device)

        # ---- Train D ----
        D.zero_grad(set_to_none=True)
        out_real = D(real)
        loss_real = criterion(out_real, real_y)

        z = torch.randn(b, NZ, 1, 1, device=device)
        fake = G(z)
        out_fake = D(fake.detach())
        loss_fake = criterion(out_fake, fake_y)

        lossD = loss_real + loss_fake
        lossD.backward()
        optD.step()

        # ---- Train G ----
        G.zero_grad(set_to_none=True)
        out_fake2 = D(fake)
        lossG = criterion(out_fake2, real_y)
        lossG.backward()
        optG.step()

        lossG_epoch = float(lossG.item())
        lossD_epoch = float(lossD.item())

    G_losses.append(lossG_epoch)
    D_losses.append(lossD_epoch)

    if epoch == 1 or epoch % 10 == 0:
        print(f'Epoch {epoch:04d}/{NUM_EPOCHS} | lossD={lossD_epoch:.4f} | lossG={lossG_epoch:.4f}')

    if epoch in SAVE_EPOCHS:
        save_grid(epoch)
        ckpt = {
            'epoch': epoch,
            'G': G.state_dict(),
            'D': D.state_dict(),
            'optG': optG.state_dict(),
            'optD': optD.state_dict(),
            'config': {
                'IMAGE_SIZE': IMAGE_SIZE, 'BATCH_SIZE': BATCH_SIZE,
                'NZ': NZ, 'NGF': NGF, 'NDF': NDF, 'LR': LR, 'BETAS': BETAS,
                'SEED': SEED
            }
        }
        ckpt_path = os.path.join(OUT_DIR, 'checkpoints', f'dcgan_epoch_{epoch:04d}.pth')
        torch.save(ckpt, ckpt_path)
        print('Saved checkpoint:', ckpt_path)

t1 = time.time()
print('Done. Minutes:', round((t1 - t0) / 60, 2))


In [ ]:
# --- 6) Plot loss curves ---
plt.figure(figsize=(8,5))
plt.plot(G_losses, label='G loss')
plt.plot(D_losses, label='D loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('DCGAN training losses')
plt.legend()
plt.grid(True)
outp = os.path.join(OUT_DIR, 'loss_curves.png')
plt.savefig(outp, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', outp)


In [ ]:
# --- 7) Generate final 1000 samples (for evaluation) ---
GEN_DIR = os.path.join(OUT_DIR, 'final_1000')
os.makedirs(GEN_DIR, exist_ok=True)

N = 1000
B = 64
G.eval()
idx = 0
with torch.no_grad():
    for _ in range(math.ceil(N / B)):
        b = min(B, N - idx)
        z = torch.randn(b, NZ, 1, 1, device=device)
        fake = G(z).detach().cpu()
        fake01 = (fake + 1) / 2
        for j in range(b):
            fp = os.path.join(GEN_DIR, f'gen_{idx:04d}.png')
            vutils.save_image(fake01[j], fp)
            idx += 1

print('Generated:', idx, 'images at', GEN_DIR)


## 8) Run Evaluation Scripts

These calls assume your repo has:
- `evaluation/ssim_protocol.py`
- `evaluation/symmetry_score.py`
- `evaluation/fid_evaluation.py`
- `evaluation/diversity_score.py`


In [ ]:
# --- 8A) SSIM Protocol ---
!python ../evaluation/ssim_protocol.py \
  --real_dir ../data/warli_dataset/man \
  --gen_dir  ../results/notebook_runs/final_1000 \
  --n_gen 100 --k_real 5 --size 128 --seed 42 \
  --best_of 20 \
  --out_csv ../results/notebook_runs/ssim_summary.csv


In [ ]:
# --- 8B) Symmetry ---
!python ../evaluation/symmetry_score.py \
  --img_dir ../results/notebook_runs/final_1000 \
  --size 128 --threshold 0.5 --top_k 25 \
  --out_csv ../results/notebook_runs/symmetry_summary.csv


In [ ]:
# --- 8C) FID (requires torchmetrics) ---
!python ../evaluation/fid_evaluation.py \
  --real_dir ../data/warli_dataset/man \
  --gen_dir  ../results/notebook_runs/final_1000 \
  --batch_size 32 --size 128 \
  --out_csv ../results/notebook_runs/fid_summary.csv


In [ ]:
# --- 8D) Diversity (requires scikit-learn) ---
!python ../evaluation/diversity_score.py \
  --img_dir ../results/notebook_runs/final_1000 \
  --batch_size 32 --size 128 \
  --out_csv ../results/notebook_runs/diversity_summary.csv


## 9) Quick Figure: Real vs Generated (Convergence)

Creates a side-by-side panel for the paper.


In [ ]:
# --- 9) Real vs Generated panel (5x5 grids) ---
from PIL import Image

fig_out = os.path.join(OUT_DIR, 'Fig_real_vs_generated_convergence.png')

# Real grid
real_loader = DataLoader(dataset, batch_size=25, shuffle=True, drop_last=True)
real_batch, _ = next(iter(real_loader))
real01 = (real_batch + 1) / 2
grid_real = vutils.make_grid(real01, nrow=5, padding=2)

# Fake grid
with torch.no_grad():
    z = torch.randn(25, NZ, 1, 1, device=device)
    fake = G(z).detach().cpu()
fake01 = (fake + 1) / 2
grid_fake = vutils.make_grid(fake01, nrow=5, padding=2)

def chw_to_hwc(x):
    return x.permute(1,2,0).numpy()

plt.figure(figsize=(14,7))
fig, axes = plt.subplots(1,2, figsize=(14,7))
axes[0].imshow(chw_to_hwc(grid_real), cmap='gray'); axes[0].set_title('Real Warli motifs'); axes[0].axis('off')
axes[1].imshow(chw_to_hwc(grid_fake), cmap='gray'); axes[1].set_title('Generated (Convergence)'); axes[1].axis('off')
plt.tight_layout()
plt.savefig(fig_out, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', fig_out)
